
### Objective:
##### Ingest YouTube trending data from multiple countries, perform data validation and quality checks, and store the raw data in a Bronze Delta Lake layer.

#### Step 1: Import Libraries

In [0]:
from pyspark.sql.functions import lit, expr, col

#### Step 2: Initial Data Exploration

In [0]:
df_preview = spark.read.csv(
    "/Volumes/workspace/default/youtube_data/INvideos.csv",
    header=True,
    inferSchema=True
)

display(df_preview.limit(5))
df_preview.printSchema()

video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
kzwfHumJyYc,17.14.11,Sharry Mann: Cute Munda ( Song Teaser) | Parmish Verma | Releasing on 17 November,Lokdhun Punjabi,1,2017-11-12T12:20:39.000Z,"""sharry mann|""""sharry mann new song""""|""""sharry mann cute munda""""|""""sharry mann latest song""""|""""sharry mann punjabi song 2017""""|""""parmish verma""""|""""parmish verma new song""""|""""parmish verma sharry mann""""|""""parmish verma sharry mann new song""""|""""parmish verma cute munda""""|""""new punjabi song 2017""""|""""punjabi song 2017""""|""""parmish verma new song 2017""""|""""parmish verma latest song 2017""""|""""punjabi songs 2017""""""",1096327,33966,798,882,https://i.ytimg.com/vi/kzwfHumJyYc/default.jpg,false,false,false,"Presenting Sharry Mann latest Punjabi Song Cute Munda Teaser . The music of new punjabi song is given by Gift Rulers while lyrics are penned by Zaildar Pargat Singh. The video is directed by Parmish Verma. \nEnjoy and stay connected with us!! \n\nSong : Cute Munda\nSinger : Sharry Mann\nStarring : Sharry Mann, Rumman & Parmish Verma\nMusic : Gift Rulers\nLyrics : Zaildar Pargat Singh\nConcept, Screenplay & Direction : Parmish Verma\nOnline Promotions : Gold Media\nCopyright: Lokdhun\n\nFull Song Releasing on 17th November\n\n\nFor more new Punjabi songs, latest Punjabi videos, funny Punjabi comedy scenes and new Punjabi movies, subscribe our channel - http://goo.gl/NnoXVB\n\n\nLike us on Facebook - https://www.facebook.com/LokdhunPunjabiOfficial/\nFollow us on Twitter - https://twitter.com/lokdhunpunjabi\nFollow us on Instagram - https://www.instagram.com/lokdhunpunjabi\nVisit us on https://www.lokdhun.com"
zUZ1z7FwLc8,17.14.11,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दंग रह जायेंगे",HJ NEWS,25,2017-11-13T05:43:56.000Z,"""पीरियड्स के समय|""""पेट पर पति करता ऐसा""""|""""देखकर दंग रह जायेंगे""""|""""latest news""""|""""today news""""|""""news""""|""""breaking news""""|""""current news""""|""""world news""""|""""hj news updates""""|""""bollywood updates""""|""""news channel in hindi""""|""""entertainment news""""|""""merrage""""|""""love""""|""""break up""""|""""perideus time""""|""""pragenent girl""""|""""baby born""""""",590101,735,904,0,https://i.ytimg.com/vi/zUZ1z7FwLc8/default.jpg,true,false,false,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दंग रह जायेंगे \n\nWatch this video :- https://youtu.be/zUZ1z7FwLc8\n\nHere is the secret of the death of PCS wife Namrata on February 16, 2017, still persisted after 9 months. In the case, the relatives of the deceased had accused the in-laws of torture against dowry, after which the police sent the accused husband and mother-in-law and sent them to jail.\n\nThe mother of the deceased Kiran said, Humility was my little daughter, she was preparing to become an IAS. Then there was a relation of dedication from a matrimonial website. We were watching the relationship for our eldest daughter, I liked the humility. \n- We did not want that little sister's marriage to be a big one before, but humility itself was refusing marriage, but Mother of Divine took it to the center. Dipartan was PCS, so we also agreed for marriage. . ''\nDeepa's mother Anuradha had said that humility could continue her studies even after marriage. On this condition she was ready for marriage, both of them were married on June 10, 2015.\n- Cajun Sister Chitra said, The sister was emotionally attached to the lamp, there were many things that she used to share with me, but Deep used to torture her for dowry.\n- During the periods, the lamp beat him on the stomach and back. Sister came home and asked me to massage the stomach and back. She used to tell that Deep struck a lot, massage it. \n- I had advised sister to take the Divorce, but she did not want to break the relationship.\n\nSubscribe Us for Latest News & Updates ► https://goo.gl/K

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: boolean (nullable = true)
 |-- ratings_disabled: boolean (nullable = true)
 |-- video_error_or_removed: boolean (nullable = true)
 |-- description: string (nullable = true)



##### Observation:
###### The YouTube dataset contains multiline descriptions, quoted text, and malformed records that may not be handled correctly by the default CSV reader.

###### To ensure reliable ingestion, a production-ready CSV parser configuration is used below.

#### Step 3: Production Data Ingestion

##### Read Data for IN:

In [0]:
df_in = spark.read \
.option("header", "true") \
.option("multiLine", "true") \
.option("quote", '"') \
.option("escape", '"') \
.option("mode", "PERMISSIVE") \
.csv("/Volumes/workspace/default/youtube_data/INvideos.csv") \
.withColumn("country", lit("IN"))

##### for US:

In [0]:
df_us = spark.read \
.option("header", "true") \
.option("multiLine", "true") \
.option("quote", '"') \
.option("escape", '"') \
.option("mode", "PERMISSIVE") \
.csv("/Volumes/workspace/default/youtube_data/USvideos.csv") \
.withColumn("country", lit("US"))

##### for GB:

In [0]:
df_gb = spark.read \
.option("header", "true") \
.option("multiLine", "true") \
.option("quote", '"') \
.option("escape", '"') \
.option("mode", "PERMISSIVE") \
.csv("/Volumes/workspace/default/youtube_data/GBvideos.csv") \
.withColumn("country", lit("GB"))

##### for CA:

In [0]:
df_ca = spark.read \
.option("header", "true") \
.option("multiLine", "true") \
.option("quote", '"') \
.option("escape", '"') \
.option("mode", "PERMISSIVE") \
.csv("/Volumes/workspace/default/youtube_data/CAvideos.csv") \
.withColumn("country", lit("CA"))

#### Step 4: Merge Data

In [0]:
youtube_df = (
    df_in
    .unionByName(df_us)
    .unionByName(df_gb)
    .unionByName(df_ca)
)

##### Verification:

In [0]:
display(
    youtube_df.select(
        "video_id",
        "channel_title",
        "country"
    )
)

video_id,channel_title,country
kzwfHumJyYc,Lokdhun Punjabi,IN
zUZ1z7FwLc8,HJ NEWS,IN
10L1hZ9qa58,TFPC,IN
N1vE8iiEg64,Eruma Saani,IN
kJzGH0PVQHQ,Filmylooks,IN
il_pSa5l98w,Dil Raju,IN
7MxiQ4v0EnE,Speed Records,IN
c64I9HNpiOY,T-Series,IN
KObFEYCaRx8,Top Telugu Media,IN
g8QsfJhFpjY,Jump Cuts,IN


#### Step 5: Data Validation

In [0]:
print("Rows:", youtube_df.count())

Rows: 158098


In [0]:
print("Columns:", len(df_in.columns))

Columns: 17


In [0]:
display(youtube_df.limit(10))

video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
kzwfHumJyYc,17.14.11,Sharry Mann: Cute Munda ( Song Teaser) | Parmish Verma | Releasing on 17 November,Lokdhun Punjabi,1,2017-11-12T12:20:39.000Z,"sharry mann|""sharry mann new song""|""sharry mann cute munda""|""sharry mann latest song""|""sharry mann punjabi song 2017""|""parmish verma""|""parmish verma new song""|""parmish verma sharry mann""|""parmish verma sharry mann new song""|""parmish verma cute munda""|""new punjabi song 2017""|""punjabi song 2017""|""parmish verma new song 2017""|""parmish verma latest song 2017""|""punjabi songs 2017""",1096327,33966,798,882,https://i.ytimg.com/vi/kzwfHumJyYc/default.jpg,FALSE,FALSE,FALSE,"Presenting Sharry Mann latest Punjabi Song Cute Munda Teaser . The music of new punjabi song is given by Gift Rulers while lyrics are penned by Zaildar Pargat Singh. The video is directed by Parmish Verma. \nEnjoy and stay connected with us!! \n\nSong : Cute Munda\nSinger : Sharry Mann\nStarring : Sharry Mann, Rumman & Parmish Verma\nMusic : Gift Rulers\nLyrics : Zaildar Pargat Singh\nConcept, Screenplay & Direction : Parmish Verma\nOnline Promotions : Gold Media\nCopyright: Lokdhun\n\nFull Song Releasing on 17th November\n\n\nFor more new Punjabi songs, latest Punjabi videos, funny Punjabi comedy scenes and new Punjabi movies, subscribe our channel - http://goo.gl/NnoXVB\n\n\nLike us on Facebook - https://www.facebook.com/LokdhunPunjabiOfficial/\nFollow us on Twitter - https://twitter.com/lokdhunpunjabi\nFollow us on Instagram - https://www.instagram.com/lokdhunpunjabi\nVisit us on https://www.lokdhun.com",IN
zUZ1z7FwLc8,17.14.11,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दंग रह जायेंगे",HJ NEWS,25,2017-11-13T05:43:56.000Z,"पीरियड्स के समय|""पेट पर पति करता ऐसा""|""देखकर दंग रह जायेंगे""|""latest news""|""today news""|""news""|""breaking news""|""current news""|""world news""|""hj news updates""|""bollywood updates""|""news channel in hindi""|""entertainment news""|""merrage""|""love""|""break up""|""perideus time""|""pragenent girl""|""baby born""",590101,735,904,0,https://i.ytimg.com/vi/zUZ1z7FwLc8/default.jpg,TRUE,FALSE,FALSE,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दंग रह जायेंगे \n\nWatch this video :- https://youtu.be/zUZ1z7FwLc8\n\nHere is the secret of the death of PCS wife Namrata on February 16, 2017, still persisted after 9 months. In the case, the relatives of the deceased had accused the in-laws of torture against dowry, after which the police sent the accused husband and mother-in-law and sent them to jail.\n\nThe mother of the deceased Kiran said, Humility was my little daughter, she was preparing to become an IAS. Then there was a relation of dedication from a matrimonial website. We were watching the relationship for our eldest daughter, I liked the humility. \n- We did not want that little sister's marriage to be a big one before, but humility itself was refusing marriage, but Mother of Divine took it to the center. Dipartan was PCS, so we also agreed for marriage. . ''\nDeepa's mother Anuradha had said that humility could continue her studies even after marriage. On this condition she was ready for marriage, both of them were married on June 10, 2015.\n- Cajun Sister Chitra said, The sister was emotionally attached to the lamp, there were many things that she used to share with me, but Deep used to torture her for dowry.\n- During the periods, the lamp beat him on the stomach and back. Sister came home and asked me to massage the stomach and back. She used to tell that Deep struck a lot, massage it. \n- I had advised sister to take the Divorce, but she did not want to break the relationship.\n\nSubscribe Us for Latest News & Updates ► https://goo.gl/K8pjDQ\n\nStay Connected with Us :\n\nFollow Us On Facebook ► https://goo.gl/GEkOqP\n\nFollow Us On Twitter ► https://twitter.

In [0]:
youtube_df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- country: string (nullable = false)



#### Step 6: Fix Data Types

In [0]:

youtube_df = youtube_df \
.withColumn("views", expr("try_cast(views as BIGINT)")) \
.withColumn("likes", expr("try_cast(likes as BIGINT)")) \
.withColumn("dislikes", expr("try_cast(dislikes as BIGINT)")) \
.withColumn("comment_count", expr("try_cast(comment_count as BIGINT)"))

In [0]:
youtube_df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: long (nullable = true)
 |-- likes: long (nullable = true)
 |-- dislikes: long (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- country: string (nullable = false)



#### Step 7: Remove Invalid Records

In [0]:
youtube_df = youtube_df.filter(
    "video_id IS NOT NULL"
)

In [0]:
youtube_df = youtube_df.filter(
    "views IS NOT NULL"
)

#### Step 8: Bronze Layer Creation

In [0]:
bronze_path = "/Volumes/workspace/default/youtube_data/bronze"

youtube_df.write \
.format("delta") \
.mode("overwrite") \
.save(bronze_path)

#### Step 9: Verification:

In [0]:
bronze_df = spark.read \
.format("delta") \
.load(bronze_path)

print("Rows:", bronze_df.count())

display(bronze_df.limit(10))

Rows: 158098


video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
Jw1Y-zhQURU,17.14.11,John Lewis Christmas Ad 2017 - #MozTheMonster,John Lewis,26,2017-11-10T07:38:29.000Z,"""christmas""|""john lewis christmas""|""john lewis""|""christmas ad""|""mozthemonster""|""christmas 2017""|""christmas ad 2017""|""john lewis christmas advert""|""moz""",7224515,55681,10247,9479,https://i.ytimg.com/vi/Jw1Y-zhQURU/default.jpg,False,False,False,"Click here to continue the story and make your own monster:\nhttp://bit.ly/2mboXgj\n\nJoe befriends a noisy Monster under his bed but the two have so much fun together that he can't get to sleep, leaving him tired by day. For Christmas Joe receives a gift to help him finally get a good night’s sleep.\n\nShop the ad\nhttp://bit.ly/2hg04Lc\n\nThe music is Golden Slumbers performed by elbow, the original song was by The Beatles. \nFind the track:\nhttps://Elbow.lnk.to/GoldenSlumbersXS\n\nSubscribe to this channel for regular video updates\nhttp://bit.ly/2eU8MvW\n\nIf you want to hear more from John Lewis:\n\nLike John Lewis on Facebook\nhttp://www.facebook.com/johnlewisretail\n\nFollow John Lewis on Twitter\nhttp://twitter.com/johnlewisretail\n\nFollow John Lewis on Instagram\nhttp://instagram.com/johnlewisretail",GB
3s1rvMFUweQ,17.14.11,Taylor Swift: …Ready for It? (Live) - SNL,Saturday Night Live,24,2017-11-12T06:24:44.000Z,"""SNL""|""Saturday Night Live""|""SNL Season 43""|""Episode 1730""|""Tiffany Haddish""|""Taylor Swift""|""Taylor Swift Ready for It""|""s43""|""s43e5""|""episode 5""|""live""|""new york""|""comedy""|""sketch""|""funny""|""hilarious""|""late night""|""host""|""music""|""guest""|""laugh""|""impersonation""|""actor""|""improv""|""musician""|""comedian""|""actress""|""If Loving You Is Wrong""|""Oprah Winfrey""|""OWN""|""Girls Trip""|""The Carmichael Show""|""Keanu""|""Reputation""|""Look What You Made Me Do""|""ready for it?""",1053632,25561,2294,2757,https://i.ytimg.com/vi/3s1rvMFUweQ/default.jpg,False,False,False,Musical guest Taylor Swift performs …Ready for It? on Saturday Night Live.\n\n#SNL #SNL43\n\nGet more SNL: http://www.nbc.com/saturday-night-live\nFull Episodes: http://www.nbc.com/saturday-night-liv...\n\nLike SNL: https://www.facebook.com/snl\nFollow SNL: https://twitter.com/nbcsnl\nSNL Tumblr: http://nbcsnl.tumblr.com/\nSNL Instagram: http://instagram.com/nbcsnl \nSNL Pinterest: http://www.pinterest.com/nbcsnl/,GB
n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"""Eminem""|""Walk""|""On""|""Water""|""Aftermath/Shady/Interscope""|""Rap""",17158579,787420,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé is available everywhere: http://shady.sr/WOWEminem \nPlaylist Best of Eminem: https://goo.gl/AquNpo\nSubscribe for more: https://goo.gl/DxCrDV\n\nFor more visit: \nhttp://eminem.com\nhttp://facebook.com/eminem\nhttp://twitter.com/eminem\nhttp://instagram.com/eminem\nhttp://eminem.tumblr.com\nhttp://shadyrecords.com\nhttp://facebook.com/shadyrecords\nhttp://twitter.com/shadyrecords\nhttp://instagram.com/shadyrecords\nhttp://trustshady.tumblr.com\n\nMusic video by Eminem performing Walk On Water. (C) 2017 Aftermath Records\nhttp://vevo.ly/gA7xKt,GB
PUTEiSjKwJU,17.14.11,Goals from Salford City vs Class of 92 and Friends at The Peninsula Stadium!,Salford City Football Club,17,2017-11-13T02:30:38.000Z,"""Salford City FC""|""Salford City""|""Salford""|""Class of 92""|""University of Salford""|""Salford Uni""|""Non League""|""National League""|""National League North""",27833,193,12,37,https://i.ytimg.com/vi/PUTEiSjKwJU/default.jpg,False,False,False,Salford drew 4-4 against the Class of 92 and Friends at the newly opened The Peninsula Stadium!\n\nLike us on Facebook: https://www.facebook.com/SalfordCityFC/ \nFollow us on 